In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install lifelines -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 16.8 MB/s eta 0:00:00


In [3]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold

Define paths and constants

In [4]:
UNICOX_OUTPUT_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/"
MODEL_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Model/UnivariateCox/"
RESULTS_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/UnivariateCox/"

os.makedirs(UNICOX_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = UNICOX_OUTPUT_DIR + "hcc_UniCox_train_selected.csv"
TEST_PATH = UNICOX_OUTPUT_DIR + "hcc_UniCox_test_selected.csv"

MODEL_PATH = MODEL_DIR + "hcc_UniCox_coxph_model.pkl"
CV_RESULTS_PATH = RESULTS_DIR + "hcc_UniCox_coxph_cv_results.csv"
CV_DETAILS_PATH = RESULTS_DIR + "hcc_UniCox_coxph_cv_fold_details.csv"

DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"

PENALIZER_GRID = [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
L1_RATIO_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]

N_SPLITS = 5
RANDOM_STATE = 42

Load selected train and test data

In [5]:
train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

train_data.shape, test_data.shape


((1085, 52), (272, 52))

In [6]:
test_data.head()

,METASTASIS_Yes,CURATIVE_INTENT_SURGERY_Yes,CURATIVE_INTENT_SURGERY_No,METASTASIS_No,MAJOR_LIVER_RESECTION_,RESECTION_MARGIN_,RESECTION_MARGIN_R0,C_N_Yes,PNI_,LVI_,...,TUMOR_STAGE_Poorly differentiated,TUMOR_STAGE_Well differentiated,MUT_SHQ1,CNA_XPO1,CNA_REL,MUT_KRAS,NEOADJUVANT_SYSTEMIC_THERAPY_No,MUT_FGFR4,OS_MONTHS,OS_EVENT
0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,28.980,1
1,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,5.952,1
2,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,9.984,0
3,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,23.652,1
4,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,36.696,0


Basic checks

In [7]:
print("Train missing values:", int(train_data.isna().sum().sum()))
print("Test missing values:", int(test_data.isna().sum().sum()))

print("\nTrain OS_EVENT distribution:")
print(train_data[EVENT_COL].value_counts())

print("\nTest OS_EVENT distribution:")
print(test_data[EVENT_COL].value_counts())

Train missing values: 0
Test missing values: 0

Train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64

Test OS_EVENT distribution:
OS_EVENT
1    161
0    111
Name: count, dtype: int64


ID leakage guard

In [8]:
for id_col in ["PATIENT_ID", "SAMPLE_ID"]:
    assert id_col not in train_data.columns, f"{id_col} found in train_data"
    assert id_col not in test_data.columns, f"{id_col} found in test_data"

print("ID leakage guard passed.")


ID leakage guard passed.


Optional drop block

In [9]:
COLS_TO_DROP = []

if COLS_TO_DROP:
    train_data = train_data.drop(columns=COLS_TO_DROP)
    test_data = test_data.drop(columns=COLS_TO_DROP)

train_data.shape, test_data.shape


((1085, 52), (272, 52))

Stratified CV helper

In [10]:
def apply_cross_validation(df, penalty, l1_ratio, duration_col, event_col, n_splits=5, random_state=42):
    X = df.reset_index(drop=True)
    y = X[event_col].astype(int).values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_details = []
    c_index_scores = []

    for fold_idx, (train_index, val_index) in enumerate(skf.split(X, y), start=1):
        train_fold = X.iloc[train_index].copy()
        val_fold = X.iloc[val_index].copy()

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                cph = CoxPHFitter(penalizer=penalty, l1_ratio=l1_ratio)
                cph.fit(train_fold, duration_col=duration_col, event_col=event_col)

            val_predictions = cph.predict_partial_hazard(val_fold)
            c_index = concordance_index(
                val_fold[duration_col],
                -val_predictions,
                val_fold[event_col]
            )

            c_index_scores.append(float(c_index))
            fold_details.append({
                "penalizer": penalty,
                "l1_ratio": l1_ratio,
                "fold": fold_idx,
                "status": "success",
                "c_index": float(c_index),
                "error_message": ""
            })

        except Exception as e:
            fold_details.append({
                "penalizer": penalty,
                "l1_ratio": l1_ratio,
                "fold": fold_idx,
                "status": "failed",
                "c_index": np.nan,
                "error_message": str(e)
            })

    return c_index_scores, fold_details


Grid search

In [11]:
cv_results = []
cv_fold_details = []

for penalty in PENALIZER_GRID:
    for l1_ratio in L1_RATIO_GRID:
        scores, details = apply_cross_validation(
            df=train_data,
            penalty=penalty,
            l1_ratio=l1_ratio,
            duration_col=DURATION_COL,
            event_col=EVENT_COL,
            n_splits=N_SPLITS,
            random_state=RANDOM_STATE
        )

        cv_fold_details.extend(details)

        successful_folds = len(scores)
        failed_folds = N_SPLITS - successful_folds

        cv_results.append({
            "penalizer": penalty,
            "l1_ratio": l1_ratio,
            "successful_folds": successful_folds,
            "failed_folds": failed_folds,
            "mean_c_index": np.mean(scores) if successful_folds > 0 else np.nan,
            "std_c_index": np.std(scores) if successful_folds > 0 else np.nan,
            "status": "ok" if successful_folds > 0 else "all_failed"
        })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    ["mean_c_index", "successful_folds"],
    ascending=[False, False]
).reset_index(drop=True)

cv_fold_details_df = pd.DataFrame(cv_fold_details)

cv_results_df.head()


,penalizer,l1_ratio,successful_folds,failed_folds,mean_c_index,std_c_index,status
0,0.10,0.00,3,2,0.737662,0.008617,ok
1,0.05,0.00,3,2,0.737023,0.009169,ok
2,0.30,0.00,3,2,0.736185,0.005679,ok
3,0.05,0.25,3,2,0.735689,0.010027,ok
4,0.05,0.50,3,2,0.735242,0.006907,ok


Save CV diagnostics

In [12]:
cv_results_df.to_csv(CV_RESULTS_PATH, index=False)
cv_fold_details_df.to_csv(CV_DETAILS_PATH, index=False)

CV_RESULTS_PATH, CV_DETAILS_PATH


('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/UnivariateCox/hcc_UniCox_coxph_cv_results.csv',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/UnivariateCox/hcc_UniCox_coxph_cv_fold_details.csv')

Select best setting

In [13]:
valid_results = cv_results_df.dropna(subset=["mean_c_index"]).copy()
best_penalty = valid_results.loc[0, "penalizer"]
best_l1_ratio = valid_results.loc[0, "l1_ratio"]
best_cv_c_index = valid_results.loc[0, "mean_c_index"]

print("Best penalizer:", best_penalty)
print("Best l1_ratio:", best_l1_ratio)
print("Best mean validation C-index:", round(best_cv_c_index, 4))


Best penalizer: 0.1
Best l1_ratio: 0.0
Best mean validation C-index: 0.7377


Train final model

In [14]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cph_final = CoxPHFitter(penalizer=best_penalty, l1_ratio=best_l1_ratio)
    cph_final.fit(train_data, duration_col=DURATION_COL, event_col=EVENT_COL)

cph_final.print_summary()


<lifelines.CoxPHFitter: fitted with 1085 total observations, 444 right-censored observations>
             duration col = 'OS_MONTHS'
                event col = 'OS_EVENT'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 1085
number of events observed = 641
   partial log-likelihood = -3722.09
         time fit was run = 2026-06-28 12:41:00 UTC

---
                                   coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                          
METASTASIS_Yes                     0.31      1.36      0.14            0.03            0.59                1.03                1.81
CURATIVE_INTENT_SURGERY_Yes       -0.19      0.82      0.17           -0.53            0.14                0.59                1.15
CURATIVE_INTENT_SURGERY_No         0.17      1.18      0.17           -0.16            0.50                0.85                1.64
METASTASIS_No                     -0.13      0.88      0.14           -0.41            0.15                0.66                1.16
MAJOR_LIVER_RESECTION_             0.05      1.05      0.17           -0.29            0.38                0.75                1.47
RESECTION_MARGIN_                  0.05      1.05      0.16           -0.27            0.37                0.76                1.44
RESECTION_MARGIN_R0               -0.25      0.78      0.13           -0.50            0.01                0.60                1.01
C_N_Yes                            0.15      1.16      0.12           -0.09            0.38                0.92                1.46
PNI_                               0.05      1.05      0.16           -0.26            0.36                0.77                1.44
LVI_                              -0.08      0.93      0.16           -0.39            0.24                0.68                1.26
C_N_No                            -0.06      0.94      0.11           -0.28            0.16                0.75                1.17
PALLIATIVE_SYSTEMIC_THERPAY_Yes    0.17      1.18      0.15           -0.13            0.47                0.87                1.59
PNI_No                            -0.11      0.89      0.16           -0.43            0.21                0.65                1.24
PALLIATIVE_SYSTEMIC_THERPAY_No    -0.11      0.90      0.15           -0.41            0.19                0.66                1.21
MAJOR_LIVER_RESECTION_No          -0.04      0.96      0.16           -0.36            0.28                0.70                1.33
MULTIFOCAL_DISEASE_Yes             0.06      1.06      0.13           -0.20            0.32                0.82                1.37
MULTIFOCAL_DISEASE_No              0.01      1.01      0.13           -0.25            0.27                0.78                1.31
LVI_No                            -0.04      0.96      0.17           -0.38            0.30                0.68                1.34
P_N_Negative                      -0.22      0.80      0.11           -0.43           -0.00                0.65                1.00
ADJUVANT_SYSTEMIC_THERAPY_No       0.12      1.13      0.18           -0.22            0.46                0.80                1.59
ADJUVANT_SYSTEMIC_THERAPY_Yes     -0.04      0.96      0.18           -0.40            0.31                0.67                1.36
MAJOR_LIVER_RESECTION_Yes         -0.04      0.97      0.18           -0.39            0.32                0.68                1.37
P_N_                              -0.04      0.96      0.10           -0.24            0.15                0.79                1.16
CNA_CDKN2A                        -0.11      0.90      0.13           -0.36            0.14                0.70                1.16
CNA_TP53                          -1.24      0.29      0.27           -1.77           -0.71                0.17                0.49
CNA_

Training C-index

In [15]:
train_c_index = cph_final.concordance_index_
print(f"Training Concordance Index: {train_c_index:.4f}")


Training Concordance Index: 0.7471


Quick test check

In [16]:
test_predictions = cph_final.predict_partial_hazard(test_data)
test_c_index = concordance_index(
    test_data[DURATION_COL],
    -test_predictions,
    test_data[EVENT_COL]
)

print(f"Test Concordance Index: {test_c_index:.4f}")


Test Concordance Index: 0.7253


Coefficient summary

In [17]:
cox_summary = cph_final.summary[["coef", "exp(coef)", "p"]].sort_values("p")
cox_summary.head(15)


,coef,exp(coef),p
covariate,,,
CNA_TP53,-1.242123,0.288770,0.000004
CNA_AXL,1.796440,6.028150,0.000044
MUT_SHQ1,1.576031,4.835724,0.000775
DIAGNOSIS_AGE,0.010732,1.010790,0.001535
CNA_POLD1,5.000000,148.413157,0.001565
CNA_PPP2R1A,5.000000,148.413157,0.001565
MUT_CD276,2.419085,11.235573,0.002579
MUT_FGFR4,1.732219,5.653183,0.005822
CNA_HOXB13,1.070315,2.916299,0.005974


Save model

In [18]:
with open(MODEL_PATH, "wb") as pickle_file:
    pickle.dump(cph_final, pickle_file)

MODEL_PATH


'/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Model/UnivariateCox/hcc_UniCox_coxph_model.pkl'

**UnivariateCox CoxPH Training Summary**

After updating the UnivariateCox feature-selection notebook, the CoxPH training notebook was rerun using the newly selected 50 features. The updated feature set was no longer selected primarily by univariate C-index strength. Instead, features were selected using FDR-adjusted CoxPH p-values as the primary criterion, with C-index strength used only as a secondary ranking criterion. This made the selected feature set more statistically grounded before model training.

The CoxPH training notebook then used these 50 selected UnivariateCox features to train a multivariable penalized Cox proportional hazards model.

**Training Process Followed**

The notebook loaded the updated UnivariateCox train and test selected datasets. The training dataset contained 1,085 patients, with 641 observed survival events. The model was trained using `lifelines.CoxPHFitter`.

A cross-validation grid search was performed over CoxPH regularization settings:

| Hyperparameter | Best Value |
|---|---:|
| Penalizer | `0.1` |
| L1 ratio | `0.0` |
| Regularization Type | Ridge-style L2 regularization |
| Best mean validation C-index | `0.7377` |

The best model used `penalizer = 0.1` and `l1_ratio = 0.0`, meaning the final CoxPH model used L2 regularization only. This is useful here because the selected features include correlated clinical categories, paired one-hot encodings, and gene alteration features. L2 regularization helps stabilize coefficients without forcing them to zero.

**Final CoxPH Model Fit**

The final CoxPH model was trained on all 1,085 training patients using the best hyperparameters from cross-validation.

| Model Detail | Value |
|---|---:|
| Model | `lifelines.CoxPHFitter` |
| Duration column | `OS_MONTHS` |
| Event column | `OS_EVENT` |
| Training observations | `1085` |
| Observed events | `641` |
| Penalizer | `0.1` |
| L1 ratio | `0.0` |
| Baseline estimation | Breslow |
| Partial log-likelihood | `-3722.09` |
| Partial AIC | `7544.18` |
| Log-likelihood ratio test | `337.62` on 50 df |
| Overall model concordance | `0.75` |

The log-likelihood ratio test was highly significant, showing that the 50-feature model as a whole provides meaningful survival information.

**Model Performance**

| Metric | Value |
|---|---:|
| Best mean CV validation C-index | `0.7377` |
| Training C-index | `0.7471` |
| Test C-index | `0.7253` |

The model performed well on the training set with a C-index of `0.7471`. On the held-out test set, the C-index was `0.7253`. This indicates that the model still has useful discriminatory ability, but performance dropped compared with training and cross-validation.

The drop from training `0.7471` to test `0.7253` suggests mild generalization loss. It is not catastrophic, but it means the updated FDR-selected feature set did not improve held-out CoxPH discrimination compared with the earlier UnivariateCox CoxPH run, where the test C-index was around `0.7472`.

**Most Important Multivariable CoxPH Predictors**

The final multivariable CoxPH model identified several statistically important predictors after accounting for all other selected features.

| Feature | Coef | HR | p-value | Interpretation |
|---|---:|---:|---:|---|
| `CNA_TP53` | `-1.2421` | `0.2888` | `0.000004` | Strong protective association in this fitted model. |
| `CNA_AXL` | `1.7964` | `6.0282` | `0.000044` | Strong risk-increasing association. |
| `MUT_SHQ1` | `1.5760` | `4.8357` | `0.000775` | Risk-increasing mutation feature. |
| `DIAGNOSIS_AGE` | `0.0107` | `1.0108` | `0.001535` | Older age modestly increases hazard. |
| `CNA_POLD1` | `5.0000` | `148.4132` | `0.001565` | Very large risk-increasing effect estimate. |
| `CNA_PPP2R1A` | `5.0000` | `148.4132` | `0.001565` | Very large risk-increasing effect estimate. |
| `MUT_CD276` | `2.4191` | `11.2356` | `0.002579` | Strong risk-increasing mutation feature. |
| `MUT_FGFR4` | `1.7322` | `5.6532` | `0.005822` | Risk-increasing mutation feature. |
| `CNA_HOXB13` | `1.0703` | `2.9163` | `0.005974` | Risk-increasing CNA feature. |
| `PRED_CATEGORY_Biliary class` | `0.3960` | `1.4859` | `0.023501` | Biliary class associated with increased hazard. |
| `MUT_KRAS` | `0.2658` | `1.3044` | `0.029892` | KRAS mutation associated with increased hazard. |
| `METASTASIS_Yes` | `0.3106` | `1.3643` | `0.031885` | Metastasis remains risk-increasing. |
| `P_N_Negative` | `-0.2193` | `0.8031` | `0.045520` | Negative pathological nodal status is protective. |

Positive coefficients indicate increased hazard, meaning worse survival. Hazard ratios greater than 1 indicate higher risk. Negative coefficients indicate lower hazard, meaning better survival. Hazard ratios below 1 indicate protective association.

**Important Clinical Predictors**

Several clinical and treatment-related features remained relevant in the model:

| Feature | Direction | Interpretation |
|---|---|---|
| `METASTASIS_Yes` | Risk-increasing | Patients with metastasis had higher predicted hazard. |
| `RESECTION_MARGIN_R0` | Protective trend | Clean surgical margin reduced hazard, although p-value was borderline at `0.0597`. |
| `P_N_Negative` | Protective | Negative nodal status was associated with lower hazard. |
| `TUMOR_SIZE` | Risk-increasing trend | Larger tumor size increased hazard slightly, borderline p-value `0.0569`. |
| `PRED_CATEGORY_Biliary class` | Risk-increasing | Biliary class was associated with worse survival. |
| `DIAGNOSIS_AGE` | Risk-increasing | Age contributed modestly but significantly to increased hazard. |

**Important Gene-Derived Predictors**

The updated FDR-based feature-selection process brought more gene-derived features into the final CoxPH model. Several became statistically significant in the multivariable setting.

Risk-increasing gene-derived predictors included:

| Feature | HR | Meaning |
|---|---:|---|
| `CNA_AXL` | `6.03` | Strongly associated with higher hazard. |
| `MUT_SHQ1` | `4.84` | Associated with higher hazard. |
| `CNA_POLD1` | `148.41` | Extremely high estimated hazard ratio. |
| `CNA_PPP2R1A` | `148.41` | Extremely high estimated hazard ratio. |
| `MUT_CD276` | `11.24` | Strongly associated with higher hazard. |
| `MUT_FGFR4` | `5.65` | Associated with higher hazard. |
| `CNA_HOXB13` | `2.92` | Associated with higher hazard. |
| `MUT_KRAS` | `1.30` | Moderately associated with higher hazard. |

Protective gene-derived predictor:

| Feature | HR | Meaning |
|---|---:|---|
| `CNA_TP53` | `0.29` | Associated with lower hazard in this fitted model. |

**Important Caution About Very Large Hazard Ratios**

`CNA_POLD1` and `CNA_PPP2R1A` both have coefficients of exactly `5.0000` and hazard ratios of `148.41`. This is a warning sign that these features may be rare, highly separated, or unstable. The confidence intervals are also very wide, from approximately `6.69` to `3291.08`.

So, while they are statistically significant in this model, they should be interpreted carefully. They are useful candidate signals, but they need validation through frequency checks, XAI comparison, and biological plausibility review.

**Comparison With Feature Selection Output**

The top feature-selection output was dominated initially by strong clinical survival variables such as:

| Feature | Univariate Direction |
|---|---|
| `METASTASIS_Yes` | Risk-increasing |
| `CURATIVE_INTENT_SURGERY_Yes` | Protective |
| `CURATIVE_INTENT_SURGERY_No` | Risk-increasing |
| `METASTASIS_No` | Protective |
| `RESECTION_MARGIN_R0` | Protective |
| `C_N_Yes` | Risk-increasing |

However, after fitting the multivariable CoxPH model, some of these features became less statistically significant. This is expected because univariate Cox evaluates one feature at a time, while multivariable Cox adjusts each feature for the presence of all other selected variables.

For example, `CURATIVE_INTENT_SURGERY_Yes` was highly significant in univariate screening but not significant in the final multivariable model. This likely happened because its effect overlaps with other treatment, surgical, metastasis, and staging-related variables.

**Main Outcome**

The updated UnivariateCox CoxPH model successfully trained using the revised statistically rigorous feature-selection strategy. The model achieved:

| Outcome | Value |
|---|---:|
| Best validation C-index | `0.7377` |
| Training C-index | `0.7471` |
| Test C-index | `0.7253` |

The model remains predictive, but its held-out test C-index is lower than the previous UnivariateCox CoxPH model. This means the revised FDR-based feature-selection method improved statistical defensibility but did not improve test discrimination for CoxPH.

**Conclusion**

This training run confirms that the updated UnivariateCox feature-selection process produces a biologically and statistically meaningful feature set. The final CoxPH model identifies both clinical and gene-derived survival predictors, including metastasis, nodal status, tumor size, age, biliary class, and several mutation/CNA features.

However, based on test C-index, this updated CoxPH model is slightly weaker than the earlier UnivariateCox CoxPH model. The advantage of this version is not necessarily higher prediction accuracy, but stronger statistical rigor and better explainability for downstream XAI and prognostic gene interpretation.